# PCA → AE → VAE 教學 Notebook

> DigitalFUTURES 2026 · 都市利害關係人工作坊 · 技術組(助教)
> 從頭到尾跑一遍:把一個城市格子的 **9 個數字**,壓成 **2 個數字**,再還原回去,最後用 VAE **生成全新格子**。

## ⚠️ 資料注意(務必先讀)
本 notebook 與所附預訓練模型,用的是 repo 內的**示範資料** `v03/cache/cells.parquet`,
**那不是最完整、最正確的原始資料**。等正式資料(由上游依 `06_INTERFACE_HANDOFF.md` 整理)到位後,
**必須把下面的 `DATA` 改成正式資料路徑、整本重跑一次**,這裡的數字、圖、模型、生成結果才算數。

## 怎麼用
由上往下,一個 cell 一個 cell 執行(Shift+Enter)。每段前面有「概念」說明,後面是可執行的程式。

## 0. 準備:載入套件與資料

我們要用三個工具箱:`numpy`(算數字)、`pandas`(讀表格)、`torch`(神經網路)。

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
%matplotlib inline
import matplotlib.pyplot as plt

# 不限制上游給幾個 channel:自動使用表格裡「所有數值欄」(排除位置/名稱/舊標籤等)。
# 要手動指定就把 CHANNELS 填成欄名 list。
CHANNELS = None
NON_CHANNEL = {"window","row","col","role","name","geometry","bid","bidx",
               "n_bld","token","token_v03","token_v04","judge","semantic","family"}

# ⚠️ 正式資料到位後,把 DATA 改成正式資料的路徑
_CAND = [os.path.join(*([".."]*k), "v03", "cache", "cells.parquet") for k in (4,3,2,1,0)]
DATA = next((p for p in _CAND if os.path.exists(p)), _CAND[0])
print("讀取資料:", DATA)

In [ ]:
# 讀表 -> 有 coverage 欄就丟空格 -> 自動決定 channel 欄位(數量不限)
df = pd.read_parquet(DATA)
if "coverage" in df.columns:
    df = df[df["coverage"] >= 0.02].reset_index(drop=True)
FEATS = CHANNELS or [c for c in df.columns
                     if c not in NON_CHANNEL and pd.api.types.is_numeric_dtype(df[c])]
X = df[FEATS].to_numpy(np.float32)        # X 形狀 [N, C]
N, C = X.shape
print(f"資料筆數 N={N}，每筆 C={C} 個數字")
print("自動選到的 channel:", FEATS)
df[FEATS].head()

### 標準化:讓 9 個數字公平競爭

`gfa` 可以到好幾萬、`coverage` 只有 0~1。不處理的話大數字會霸佔運算。
解法:每一欄各自做 `(數值 - 平均) / 標準差`,讓每欄都變成「平均 0、散開 1」。
**平均和標準差只用訓練組算**(避免偷看驗證組)。

In [ ]:
# 分訓練/驗證組(固定種子 0,結果可重現)
rng = np.random.default_rng(0)
idx = rng.permutation(N)
va = idx[:N // 5]      # 1/5 驗證
tr = idx[N // 5:]      # 4/5 訓練

# 標準化(只用 train 算 mean/std)
mean = X[tr].mean(0); std = X[tr].std(0); std[std == 0] = 1
Xz = ((X - mean) / std).astype(np.float32)
print("標準化後 平均≈%.3f  散開≈%.3f  (應接近 0 / 1)" % (Xz.mean(), Xz.std()))

def recon_err(a, b):   # 重建誤差:差距平方的平均,越小越好
    return float(np.mean((a - b) ** 2))

## 1. PCA — 用數學一步把資料壓扁

**概念**:9 維資料像一團雲。PCA 找出這團雲「最長的兩個方向」,沿著它們拍照最不失真。
- 編碼 `z = X·W`(投影到新方向)
- 解碼 `x̂ = z·Wᵀ`(從投影還原)
- PCA **不用訓練**,一次 SVD(奇異值分解)就算出最佳方向 `W`。

In [ ]:
# 對 train 做 SVD,取前 2 條主軸 W
Xc = Xz[tr] - Xz[tr].mean(0)
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
W = Vt[:2].T                       # W 形狀 [9, 2]:編碼器(就是個矩陣)
explained = (S[:2] ** 2) / (S ** 2).sum()

Z_pca = Xz @ W                     # 編碼 [N, 2]
Xhat_pca = Z_pca @ W.T             # 解碼 [N, 9]
print("兩個方向各抓住的資訊比例 =", np.round(explained, 3))
print("PCA 重建誤差  val = %.4f" % recon_err(Xz[va], Xhat_pca[va]))

In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(Z_pca[:,0], Z_pca[:,1], s=4, alpha=.4)
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("PCA latent")
plt.show()

## 2. Autoencoder(AE)— 讓神經網路自己學怎麼壓

**概念**:PCA 只能「直線」壓。把那個固定矩陣換成**會彎曲的神經網路**,讓它自己學,通常壓得更好。
代價:要「訓練」(反覆調整)。

架構:`Encoder 9→32→16→2`(瓶頸,逼資料擠過 2 維)、`Decoder 2→16→32→9`。
**瓶頸那層後面沒有激活函數**,因為壓出來的 2 個數字要能是負的。

In [ ]:
class AE(nn.Module):
    def __init__(self, c, L=2):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(c,32), nn.ReLU(),
                                 nn.Linear(32,16), nn.ReLU(), nn.Linear(16,L))
        self.dec = nn.Sequential(nn.Linear(L,16), nn.ReLU(),
                                 nn.Linear(16,32), nn.ReLU(), nn.Linear(32,c))
    def forward(self, x):
        z = self.enc(x); return self.dec(z), z

ae = AE(C)

### 訓練迴圈 = 所有神經網路的心臟(背起來)
1. **清梯度** `zero_grad()`  2. **前向** 算還原  3. **算誤差** MSE  4. **反向+更新** `backward()` / `step()`
重複幾千次,誤差往下掉 = 模型在學。

In [ ]:
opt = torch.optim.Adam(ae.parameters(), lr=1e-3)
lossf = nn.MSELoss()
Xtr = torch.tensor(Xz[tr]); Xva = torch.tensor(Xz[va])

for ep in range(1, 81):                      # 80 輪
    ae.train(); perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), 256):
        xb = Xtr[perm[i:i+256]]
        opt.zero_grad()                      # 1
        xhat, _ = ae(xb)                     # 2
        loss = lossf(xhat, xb)               # 3
        loss.backward(); opt.step()          # 4
    if ep % 20 == 0 or ep == 1:
        ae.eval()
        with torch.no_grad(): v = lossf(ae(Xva)[0], Xva).item()
        print(f"  epoch {ep:3d}  val_mse={v:.4f}")

In [ ]:
ae.eval()
with torch.no_grad():
    Xhat_ae, Z_ae = ae(torch.tensor(Xz))
Xhat_ae, Z_ae = Xhat_ae.numpy(), Z_ae.numpy()
print("AE 重建誤差  val = %.4f   (應該比 PCA 小)" % recon_err(Xz[va], Xhat_ae[va]))

plt.figure(figsize=(5,5))
plt.scatter(Z_ae[:,0], Z_ae[:,1], s=4, alpha=.4)
plt.xlabel("z1"); plt.ylabel("z2"); plt.title("AE latent")
plt.show()

> **觀察**:AE 的重建誤差通常比 PCA 小 —— 這就是「會彎的神經網路」勝過「直線 PCA」的地方。
> 但 AE **不能生成**:它的 latent 散落各處、中間有洞,隨便抽一個點解出來會是垃圾。下面用 VAE 解決。

## 3. Beta-VAE — 讓 latent 變整齊,於是能「生成」

**VAE = AE 再加三樣東西(★)**:
- **★1** 瓶頸不再吐一個點,而是吐一團分布:`μ`(中心)和 `logσ²`(散開度)。
- **★2 重參數化** `z = μ + σ·ε`:「抽籤」本身沒有梯度、模型學不動;先抽一個與模型無關的亂數 `ε`,再用 `μ、σ` 把它搬到位,梯度就能流回去。**這是 VAE 能訓練的關鍵。**
- **★3 KL 損失**:逼那團分布靠近「標準常態雲 N(0,1)」。所有資料的 latent 被收攏成一團**整齊、沒有洞**的雲 → 隨便抽都合理 → **生成**。

`β` 是旋鈕:大 = latent 整齊但重建糊;小 = 重建清楚但難生成。

In [ ]:
class VAE(nn.Module):
    def __init__(self, c, L=2):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(c,32), nn.ReLU(), nn.Linear(32,16), nn.ReLU())
        self.mu = nn.Linear(16, L)          # ★1 中心
        self.logvar = nn.Linear(16, L)      # ★1 散開度(log,保證正)
        self.dec = nn.Sequential(nn.Linear(L,16), nn.ReLU(),
                                 nn.Linear(16,32), nn.ReLU(), nn.Linear(32,c))
    def forward(self, x):
        h = self.body(x); mu = self.mu(h); logvar = self.logvar(h)
        std = torch.exp(0.5*logvar)
        z = mu + std * torch.randn_like(mu)  # ★2 重參數化
        return self.dec(z), mu, logvar

def vae_loss(xhat, x, mu, logvar, beta):
    # 重建項:對 9 個 channel「加總」再對 batch 平均(加總很重要,否則會被 KL 壓垮 -> 模型擺爛)
    recon = ((xhat - x)**2).sum(1).mean()
    kl = (-0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).sum(1)).mean()   # ★3 KL
    return recon + beta*kl, recon.item(), kl.item()

BETA = 1.0          # 🔧 試試改成 0.1 或 5,再重跑下面兩個 cell,看差別
vae = VAE(C)

In [ ]:
opt = torch.optim.Adam(vae.parameters(), lr=1e-3)
for ep in range(1, 151):                     # 150 輪
    vae.train(); perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), 256):
        xb = Xtr[perm[i:i+256]]
        opt.zero_grad()
        xhat, mu, logvar = vae(xb)
        loss, _, _ = vae_loss(xhat, xb, mu, logvar, BETA)
        loss.backward(); opt.step()
    if ep % 30 == 0 or ep == 1:
        vae.eval()
        with torch.no_grad():
            xh, mu, lv = vae(Xva); _, r, k = vae_loss(xh, Xva, mu, lv, BETA)
        print(f"  epoch {ep:3d}  recon={r:.3f}  kl={k:.3f}")

In [ ]:
# 評估重建 + 取 latent(用 μ 當座標,不抽樣,結果才穩定)
vae.eval()
with torch.no_grad():
    h = vae.body(torch.tensor(Xz)); MU = vae.mu(h)
    Xhat_vae = vae.dec(MU).numpy()
Z_vae = MU.numpy()
print("VAE 重建誤差  val = %.4f   (通常比 AE 略大,用一點重建換來能生成)" % recon_err(Xz[va], Xhat_vae[va]))

plt.figure(figsize=(5,5))
plt.scatter(Z_vae[:,0], Z_vae[:,1], s=4, alpha=.4)
plt.xlabel("z1"); plt.ylabel("z2"); plt.title(f"VAE latent (beta={BETA})")
plt.show()

### ★ 生成:從 N(0,1) 抽點 → 解碼 → 全新格子

因為 KL 已把所有資料的 latent 收進標準常態雲,雲裡到處都合理,
所以**閉著眼睛從 N(0,1) 抽幾個點、丟進解碼器,就長出全新但合理的格子**。
最後乘回 `std`、加回 `mean`,還原成真實尺度。

In [ ]:
with torch.no_grad():
    znew = torch.randn(8, 2)                 # 抽 8 個點
    gen = vae.dec(znew).numpy() * std + mean # 反標準化回真實尺度
gen_df = pd.DataFrame(gen, columns=FEATS).round(2)
print("VAE 生成的 8 筆全新格子(真實尺度):")
gen_df

## 4. 收束:三個方法一次看完

| 方法 | 怎麼壓 | 要訓練 | 重建誤差(val) | 能生成 |
|---|---|---|---|---|
| **PCA** | 線性矩陣 | 否 | 最大 | 勉強 |
| **AE** | 神經網路(會彎) | 是 | 中 | ❌(latent 有洞) |
| **VAE** | AE + 分布 + 重參數 + KL | 是 | 略大於 AE | ✅(latent 是整齊的雲) |

- **Encoder 給你分析**(真實資料 → 座標),**Decoder 給你生成**(座標 → 新資料)。VAE 兩個都有。
- 從 AE 到 VAE 只改三點:**μ/logσ² 雙頭、重參數化、KL**,其餘一字不改。

## 🔧 練習
1. 把 `BETA` 改成 `0.1` 與 `5`,重跑 VAE 的訓練與生成,觀察「重建清晰 ↔ 生成合理」的拉鋸。
2. 把 `LATENT` 從 2 改成更大(要同時改模型),看重建誤差會不會變小。

## ⚠️ 最後提醒
本 notebook 跑的是**示範資料**。**正式資料到位後,把 `DATA` 改掉、整本重跑**,數字與模型才有效。